In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split,KFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

In [2]:
pressure = pd.read_csv('./Datasets Archive/pressure.csv')
pressure = pressure['Toronto']
pressure.head()

0       NaN
1    1012.0
2    1011.0
3    1011.0
4    1010.0
Name: Toronto, dtype: float64

In [3]:
humidity = pd.read_csv('./Datasets Archive/humidity.csv')
humidity = humidity['Toronto']
humidity.head()

0     NaN
1    82.0
2    81.0
3    79.0
4    77.0
Name: Toronto, dtype: float64

In [4]:
temperature = pd.read_csv('./Datasets Archive/temperature.csv')
temperature = temperature['Toronto']
temperature = temperature - 273.15
temperature.head()

0          NaN
1    13.110000
2    13.112541
3    13.119518
4    13.126496
Name: Toronto, dtype: float64

In [5]:
wind_direction = pd.read_csv('./Datasets Archive/wind_direction.csv')
wind_direction = wind_direction['Toronto']
wind_direction.head()

0      NaN
1    240.0
2    236.0
3    226.0
4    216.0
Name: Toronto, dtype: float64

In [6]:
wind_speed = pd.read_csv('./Datasets Archive/wind_speed.csv')
wind_speed = wind_speed['Toronto']
wind_speed.head()

0    NaN
1    3.0
2    3.0
3    3.0
4    3.0
Name: Toronto, dtype: float64

In [7]:
datetime = pd.read_csv('./Datasets Archive/humidity.csv')
datetime = pd.to_datetime(datetime['datetime'])
hour = datetime.dt.hour
month = datetime.dt.month
hour.head() , month.head()


(0    12
 1    13
 2    14
 3    15
 4    16
 Name: datetime, dtype: int32,
 0    10
 1    10
 2    10
 3    10
 4    10
 Name: datetime, dtype: int32)

In [8]:
df = pd.DataFrame({
                'month':month,
                'hour':hour,
                'pressure':pressure,
                'humidity':humidity,
                'wind_direction':wind_direction,
                'wind_speed':wind_speed,
                'temperature':temperature
})
df.head()

,month,hour,pressure,humidity,wind_direction,wind_speed,temperature
0,10,12,NaN,NaN,NaN,NaN,NaN
1,10,13,1012.0,82.0,240.0,3.0,13.110000
2,10,14,1011.0,81.0,236.0,3.0,13.112541
3,10,15,1011.0,79.0,226.0,3.0,13.119518
4,10,16,1010.0,77.0,216.0,3.0,13.126496


In [9]:
df = df.dropna(subset=['temperature'])

for feature in df:
    df[feature] = df[feature].replace(0,None)
    mean_value = np.mean(df[feature])
    df[feature] = df[feature].fillna(mean_value)
df.head()

C:\Users\pc\AppData\Local\Temp\ipykernel_10144\3045584468.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[feature] = df[feature].fillna(mean_value)


,month,hour,pressure,humidity,wind_direction,wind_speed,temperature
1,10,13.0,1012.0,82.0,240.0,3.0,13.110000
2,10,14.0,1011.0,81.0,236.0,3.0,13.112541
3,10,15.0,1011.0,79.0,226.0,3.0,13.119518
4,10,16.0,1010.0,77.0,216.0,3.0,13.126496
5,10,17.0,1010.0,76.0,206.0,3.0,13.133473


In [10]:
df['hour_sin'] = np.sin(2*np.pi*df["hour"]/24)
df['hour_cos'] = np.cos(2*np.pi*df["hour"]/24)
df.drop("hour",axis=1,inplace=True)
df

,month,pressure,humidity,wind_direction,wind_speed,temperature,hour_sin,hour_cos
1,10,1012.0,82.0,240.0,3.0,13.110000,-0.258819,-0.965926
2,10,1011.0,81.0,236.0,3.0,13.112541,-0.500000,-0.866025
3,10,1011.0,79.0,226.0,3.0,13.119518,-0.707107,-0.707107
4,10,1010.0,77.0,216.0,3.0,13.126496,-0.866025,-0.500000
5,10,1010.0,76.0,206.0,3.0,13.133473,-0.965926,-0.258819
...,...,...,...,...,...,...,...,...
45248,11,1026.0,45.0,360.0,4.0,5.590000,-0.866025,0.500000
45249,11,1026.0,48.0,360.0,4.0,5.600000,-0.707107,0.707107
45250,11,1026.0,52.0,350.0,2.0,4.400000,-0.500000,0.866025
45251,11,1027.0,64.0,10.0,4.0,3.010000,-0.258819,0.965926


In [11]:
# Winter : 0
# Spring : 1
# Summer : 2
# Fall : 3

season_map = {
    1: 0, 2: 0, 3: 0,
    4: 1, 5: 1, 6: 1,
    7: 2, 8: 2, 9: 2,
    10: 3, 11: 3, 12: 3
}

df["season"] = df["month"].map(season_map)
df

,month,pressure,humidity,wind_direction,wind_speed,temperature,hour_sin,hour_cos,season
1,10,1012.0,82.0,240.0,3.0,13.110000,-0.258819,-0.965926,3
2,10,1011.0,81.0,236.0,3.0,13.112541,-0.500000,-0.866025,3
3,10,1011.0,79.0,226.0,3.0,13.119518,-0.707107,-0.707107,3
4,10,1010.0,77.0,216.0,3.0,13.126496,-0.866025,-0.500000,3
5,10,1010.0,76.0,206.0,3.0,13.133473,-0.965926,-0.258819,3
...,...,...,...,...,...,...,...,...,...
45248,11,1026.0,45.0,360.0,4.0,5.590000,-0.866025,0.500000,3
45249,11,1026.0,48.0,360.0,4.0,5.600000,-0.707107,0.707107,3
45250,11,1026.0,52.0,350.0,2.0,4.400000,-0.500000,0.866025,3
45251,11,1027.0,64.0,10.0,4.0,3.010000,-0.258819,0.965926,3


In [12]:
X  = np.array(df[["month","pressure","humidity","wind_direction","wind_speed","hour_sin","hour_cos","season"]])
y = np.array(df["temperature"])
X_train, X_test, y_train, y_test = train_test_split(X,y,train_size=0.8,random_state=12)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((36201, 8), (9051, 8), (36201,), (9051,))

In [13]:
X_scaler = StandardScaler()

X_train = X_scaler.fit_transform(X_train)
X_test = X_scaler.transform(X_test)

In [14]:
k = 10
k_fold = KFold(k,random_state=12,shuffle=True)
folds_error = []

KNN_model = KNeighborsRegressor(15)

for fold, (train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    KNN_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(KNN_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.8193027114124638


In [15]:
sgd_model = SGDRegressor()
folds_error = []

for fold, (train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    sgd_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(sgd_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.33802872968358166


In [16]:
random_forest_model = RandomForestRegressor()
folds_error = []

for fold,(train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    random_forest_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(random_forest_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.9117612247773751


In [17]:
decision_tree_model = DecisionTreeRegressor()
folds_error = []

for fold,(train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    decision_tree_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(decision_tree_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.825610282848593


In [18]:
MLP_regressor_model = MLPRegressor(max_iter=1000, random_state=12)
folds_error = []

for fold,(train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    MLP_regressor_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(MLP_regressor_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.8557573676356938


In [19]:
svr_model = SVR()
folds_error = []

for fold,(train_index,validate_index) in enumerate(k_fold.split(X_train)):
    X_train_here = X_train[train_index]
    y_train_here = y_train[train_index]

    svr_model.fit(X_train_here,y_train_here)

    X_validate = X_train[validate_index]
    y_validate = y_train[validate_index]

    folds_error.append(svr_model.score(X_validate,y_validate))

print(np.mean(folds_error))

0.8440632871477078


In [80]:
errors = []
for i in range(100,1000,200):

    random_forest_model = RandomForestRegressor(n_estimators=i)
    folds_error = []

    for fold,(train_index,validate_index) in enumerate(k_fold.split(X_train)):
        X_train_here = X_train[train_index]
        y_train_here = y_train[train_index]

        random_forest_model.fit(X_train_here,y_train_here)

        X_validate = X_train[validate_index]
        y_validate = y_train[validate_index]

        folds_error.append(random_forest_model.score(X_validate,y_validate))

    print(f"number of tree : {i}\t r2 score : {np.mean(folds_error)}")
    errors.append(np.mean(folds_error))

number of tree : 100	 r2 score : 0.8772524270846075
number of tree : 300	 r2 score : 0.8780106978845353
number of tree : 500	 r2 score : 0.8780397076604501
number of tree : 700	 r2 score : 0.8780750650357149
number of tree : 900	 r2 score : 0.8782149780698012


In [20]:
model = RandomForestRegressor()
model.fit(X_train,y_train)

RandomForestRegressor()